In [39]:
import pandas as pd

raw = pd.read_json("../data/candidates.jsonl", lines=True)

df = pd.json_normalize(
    raw.to_dict(orient="records"),
    sep="_"
)

print(df.dtypes)
print(df.isna().sum())

candidate_id                                                str
career_history                                           object
education                                                object
skills                                                   object
certifications                                           object
                                                         ...   
redrob_signals_skill_assessment_scores_Embeddings       float64
redrob_signals_skill_assessment_scores_LlamaIndex       float64
redrob_signals_skill_assessment_scores_PyTorch          float64
redrob_signals_skill_assessment_scores_Deep Learning    float64
redrob_signals_skill_assessment_scores_QLoRA            float64
Length: 95, dtype: object
candidate_id                                                0
career_history                                              0
education                                                   0
skills                                                      0
certifications        

In [40]:
df["profile_years_of_experience"].describe()

count    100000.000000
mean          7.166319
std           3.824551
min           1.000000
25%           3.900000
50%           6.800000
75%           9.900000
max          16.900000
Name: profile_years_of_experience, dtype: float64

In [41]:
def score_experience(years):
    if 5 <= years <= 9:
        return 1.0
    if (4 <= years < 5) or (9 < years <= 10):
        return 0.8
    if (3 <= years < 4) or (10 < years <= 12):
        return 0.5
    return 0.2

df["experience_score"] = df["profile_years_of_experience"].apply(score_experience)

print("Value counts:")
print(df["experience_score"].value_counts())

print("\nDescriptive statistics:")
print(df["experience_score"].describe())

Value counts:
experience_score
1.0    34375
0.2    30508
0.5    18870
0.8    16247
Name: count, dtype: int64

Descriptive statistics:
count    100000.000000
mean          0.629092
std           0.333697
min           0.200000
25%           0.200000
50%           0.800000
75%           1.000000
max           1.000000
Name: experience_score, dtype: float64


In [42]:
df["profile_current_title"].value_counts()

profile_current_title
Business Analyst                    5833
HR Manager                          5830
Mechanical Engineer                 5791
Accountant                          5764
Project Manager                     5754
Customer Support                    5750
Operations Manager                  5744
Content Writer                      5727
Sales Executive                     5713
Civil Engineer                      5702
Graphic Designer                    5689
Marketing Manager                   5524
Software Engineer                   3450
Full Stack Developer                2873
Cloud Engineer                      2836
Java Developer                      2809
.NET Developer                      2788
DevOps Engineer                     2787
Mobile Developer                    2757
Frontend Engineer                   2738
QA Engineer                         2682
Analytics Engineer                   764
Data Engineer                        744
Data Analyst                       

In [63]:
strong_title_matches = {
    "Recommendation Systems Engineer",
    "Search Engineer",
    "Machine Learning Engineer",
    "Senior Machine Learning Engineer",
    "Applied ML Engineer",
    "AI Engineer",
    "Lead AI Engineer",
    "NLP Engineer",
    "Senior NLP Engineer",
    "Senior Applied Scientist",
    "Backend Engineer",
    "Data Engineer",
    "Software Engineer",
}

medium_title_matches = {
    "DevOps Engineer",
    "Cloud Engineer",
    "Java Developer",
    "Full Stack Developer",
    ".NET Developer",
    "Mobile Developer",
    "QA Engineer",
}

weak_title_matches = {
    "Business Analyst",
    "Project Manager",
    "Operations Manager",
}

very_weak_title_matches = {
    "Accountant",
    "HR Manager",
    "Marketing Manager",
    "Graphic Designer",
    "Content Writer",
    "Customer Support",
    "Mechanical Engineer",
    "Civil Engineer",
}

title_score_map = {
    **{title: 1.0 for title in strong_title_matches},
    **{title: 0.7 for title in medium_title_matches},
    **{title: 0.3 for title in weak_title_matches},
    **{title: 0.0 for title in very_weak_title_matches},
}

df["title_score"] = df["profile_current_title"].map(title_score_map).fillna(0.0)

df["title_score"].value_counts()

title_score
0.0    58089
0.7    19532
0.3    17331
1.0     5048
Name: count, dtype: int64

In [44]:
behavior_cols = [
    "redrob_signals_open_to_work_flag",
    "redrob_signals_recruiter_response_rate",
    "redrob_signals_notice_period_days",
    "redrob_signals_interview_completion_rate",
    "redrob_signals_offer_acceptance_rate",
]

df[behavior_cols].describe(include="all")

,redrob_signals_open_to_work_flag,redrob_signals_recruiter_response_rate,redrob_signals_notice_period_days,redrob_signals_interview_completion_rate,redrob_signals_offer_acceptance_rate
count,100000,100000.000000,100000.000000,100000.000000,100000.000000
unique,2,NaN,NaN,NaN,NaN
top,False,NaN,NaN,NaN,NaN
freq,64661,NaN,NaN,NaN,NaN
mean,NaN,0.436574,87.385800,0.619510,-0.403604
std,NaN,0.214122,36.589628,0.170662,0.732439
min,NaN,0.020000,0.000000,0.300000,-1.000000
25%,NaN,0.250000,60.000000,0.480000,-1.000000
50%,NaN,0.440000,90.000000,0.620000,-1.000000
75%,NaN,0.620000,120.000000,0.760000,0.400000


In [45]:
def score_notice_period(days):
    if days <= 30:
        return 1.0
    if days <= 60:
        return 0.7
    if days <= 90:
        return 0.4
    return 0.1

df["open_to_work_score"] = df["redrob_signals_open_to_work_flag"].astype(float)
df["notice_score"] = df["redrob_signals_notice_period_days"].apply(score_notice_period)
df["offer_acceptance_score"] = df["redrob_signals_offer_acceptance_rate"].replace(-1, 0.5)

behavior_score_cols = [
    "open_to_work_score",
    "redrob_signals_recruiter_response_rate",
    "notice_score",
    "redrob_signals_interview_completion_rate",
    "offer_acceptance_score",
]

df["behavior_score"] = df[behavior_score_cols].mean(axis=1)

df["behavior_score"].describe()

count    100000.000000
mean          0.472802
std           0.129820
min           0.134000
25%           0.374000
50%           0.460000
75%           0.566000
max           0.922000
Name: behavior_score, dtype: float64

In [46]:
print(type(df.loc[0, "profile_summary"]))
print(type(df.loc[0, "career_history"]))
print(df.loc[0, "career_history"][0]["description"][:300])

<class 'str'>
<class 'list'>
Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to m


In [47]:
def safe_text(value):
    if value is None:
        return ""
    if not isinstance(value, (list, dict)) and pd.isna(value):
        return ""
    return str(value).strip()


def career_descriptions(career_history):
    if not isinstance(career_history, list):
        return ""
    return " ".join(
        safe_text(role.get("description"))
        for role in career_history
        if isinstance(role, dict) and safe_text(role.get("description"))
    )


df["candidate_text"] = (
    df["profile_summary"].apply(safe_text)
    + " "
    + df["profile_headline"].apply(safe_text)
    + " "
    + df["career_history"].apply(career_descriptions)
).str.strip()

sample_text = df.sample(3, random_state=42)[["candidate_id", "candidate_text"]].copy()
sample_text["candidate_text"] = sample_text["candidate_text"].str[:500]

print(sample_text.to_string(index=False))

candidate_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       candidate_text
CAND_0075722 Software engineer with 7.8 years of experience across web, backend, and cloud systems. Strong fundamentals in software development and system design. I've spent most of my career on web and API development — Python/Django and Node.js mostly. I've been keeping up with AI/ML at a self-learner level — taken some online courses, played with the OpenAI and Anthropic APIs, built a small RAG side project — but I haven't done it in a professional capacity yet. Open to roles wh

In [48]:
import re

keyword_groups = {
    "retrieval_keyword_count": [
        "retrieval",
        "retrieval system",
        "retrieval systems",
        "search",
        "semantic search",
        "vector search",
        "rag",
    ],
    "embedding_keyword_count": [
        "embedding",
        "embeddings",
        "semantic matching",
        "sentence transformer",
        "sentence transformers",
        "similarity search",
    ],
    "vector_db_keyword_count": [
        "vector database",
        "vector databases",
        "vector db",
        "vector store",
        "vector stores",
        "faiss",
        "pinecone",
        "weaviate",
        "milvus",
        "chroma",
        "qdrant",
    ],
    "llm_keyword_count": [
        "llm",
        "llms",
        "large language model",
        "large language models",
        "gpt",
        "openai",
        "anthropic",
        "langchain",
        "llamaindex",
        "prompt engineering",
    ],
    "evaluation_keyword_count": [
        "evaluation",
        "evaluations",
        "eval",
        "evals",
        "benchmark",
        "benchmarks",
        "metric",
        "metrics",
        "relevance scoring",
        "ranking quality",
    ],
}


def count_keywords(text, keywords):
    text = safe_text(text)
    return sum(
        len(re.findall(rf"(?<!\w){re.escape(keyword)}(?!\w)", text, flags=re.IGNORECASE))
        for keyword in keywords
    )


for count_col, keywords in keyword_groups.items():
    df[count_col] = df["candidate_text"].apply(lambda text: count_keywords(text, keywords))

keyword_count_cols = list(keyword_groups)
df["total_keyword_count"] = df[keyword_count_cols].sum(axis=1)

top_keyword_candidates = df.sort_values(
    ["total_keyword_count", "candidate_id"],
    ascending=[False, True],
).head(10)

top_keyword_candidates[["candidate_id", *keyword_count_cols, "total_keyword_count"]]


,candidate_id,retrieval_keyword_count,embedding_keyword_count,vector_db_keyword_count,llm_keyword_count,evaluation_keyword_count,total_keyword_count
41610,CAND_0041611,12,6,1,1,9,29
7411,CAND_0007412,5,6,3,6,8,28
18498,CAND_0018499,12,3,2,4,7,28
39753,CAND_0039754,12,4,2,1,8,27
46063,CAND_0046064,11,3,2,3,8,27
55904,CAND_0055905,13,3,1,3,7,27
77336,CAND_0077337,12,9,0,0,5,26
7008,CAND_0007009,7,4,3,4,7,25
81845,CAND_0081846,13,3,1,3,5,25
86021,CAND_0086022,10,6,1,2,6,25


In [49]:
top_ids = [
    "CAND_0000031",
    "CAND_0000021",
    "CAND_0000023"
]

for cid in top_ids:
    row = df[df["candidate_id"] == cid].iloc[0]

    print("=" * 100)
    print("Candidate:", cid)
    print("Title:", row["profile_current_title"])
    print("Experience:", row["profile_years_of_experience"])
    print("Company:", row["profile_current_company"])
    print()
    print(row["candidate_text"][:2000])
    print("\n")

Candidate: CAND_0000031
Title: Recommendation Systems Engineer
Experience: 6.0
Company: Swiggy

Machine learning engineer with 6.0 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I've learned that most retrieval problems are actually evaluation problems in disguise. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I'd own a meaningful piece of the ML stack. Recommendation Systems Engineer | Search, Ranking & Retrieval Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across thre

In [50]:
import re

strong_production_signals = [
    "shipped",
    "deployed",
    "in production",
    "production",
    "launched",
    "owned",
    "led",
    "designed",
    "built",
]

medium_production_signals = [
    "implemented",
    "developed",
    "worked on",
    "created",
]


def count_signal_occurrences(text, signals):
    text = safe_text(text)
    return sum(
        len(re.findall(rf"(?<!\w){re.escape(signal)}(?!\w)", text, flags=re.IGNORECASE))
        for signal in signals
    )


production_raw_score = (
    2 * df["candidate_text"].apply(lambda text: count_signal_occurrences(text, strong_production_signals))
    + df["candidate_text"].apply(lambda text: count_signal_occurrences(text, medium_production_signals))
)

max_production_raw_score = production_raw_score.max()
df["production_score"] = production_raw_score / max_production_raw_score if max_production_raw_score else 0.0

df.sort_values(
    ["production_score", "candidate_id"],
    ascending=[False, True],
).head(10)[["candidate_id", "production_score"]]


,candidate_id,production_score
19063,CAND_0019064,1.000000
43775,CAND_0043776,0.962963
49590,CAND_0049591,0.962963
61236,CAND_0061237,0.962963
62563,CAND_0062564,0.962963
2885,CAND_0002886,0.925926
4832,CAND_0004833,0.925926
24229,CAND_0024230,0.925926
47689,CAND_0047690,0.925926
79762,CAND_0079763,0.925926


In [51]:
weighted_score = (
    5 * df["retrieval_keyword_count"]
    + 4 * df["embedding_keyword_count"]
    + 4 * df["vector_db_keyword_count"]
    + 4 * df["evaluation_keyword_count"]
    + 2 * df["llm_keyword_count"]
)

max_weighted_score = weighted_score.max()
df["retrieval_score"] = weighted_score / max_weighted_score if max_weighted_score else 0.0

df.sort_values(
    ["retrieval_score", "candidate_id"],
    ascending=[False, True],
).head(10)[["candidate_id", "retrieval_score"]]


,candidate_id,retrieval_score
41610,CAND_0041611,1.000000
39753,CAND_0039754,0.936508
18498,CAND_0018499,0.920635
77336,CAND_0077337,0.920635
55904,CAND_0055905,0.912698
46063,CAND_0046064,0.896825
81845,CAND_0081846,0.849206
86021,CAND_0086022,0.841270
88024,CAND_0088025,0.841270
7411,CAND_0007412,0.833333


In [52]:
df["final_score"] = (
    0.20 * df["experience_score"]
    + 0.15 * df["title_score"]
    + 0.15 * df["behavior_score"]
    + 0.25 * df["production_score"]
    + 0.25 * df["retrieval_score"]
)
top_candidates = df.sort_values(
    "final_score",
    ascending=False
)

top_candidates[
    [
        "candidate_id",
        "profile_current_title",
        "profile_years_of_experience",
        "final_score"
    ]
].head(20)

top_candidates[
    [
        "candidate_id",
        "profile_current_title",
        "profile_years_of_experience",
        "final_score"
    ]
].head(10)

,candidate_id,profile_current_title,profile_years_of_experience,final_score
41609,CAND_0041610,Recommendation Systems Engineer,6.7,0.775187
7008,CAND_0007009,Recommendation Systems Engineer,7.9,0.763051
30,CAND_0000031,Recommendation Systems Engineer,6.0,0.713652
41668,CAND_0041669,Recommendation Systems Engineer,8.0,0.704775
18721,CAND_0018722,Recommendation Systems Engineer,6.6,0.699457
53694,CAND_0053695,Recommendation Systems Engineer,5.8,0.685690
61264,CAND_0061265,Recommendation Systems Engineer,6.6,0.681395
17959,CAND_0017960,Recommendation Systems Engineer,7.7,0.677931
77336,CAND_0077337,Staff Machine Learning Engineer,7.0,0.675029
14439,CAND_0014440,Recommendation Systems Engineer,6.4,0.665426


In [53]:
candidate = df[df["candidate_id"] == "CAND_0000036"]

candidate[
[
    "profile_current_title",
    "experience_score",
    "title_score",
    "behavior_score",
    "production_score",
    "retrieval_score",
    "final_score"
]
]
row = df[df["candidate_id"] == "CAND_0000036"].iloc[0]

print(row["candidate_text"][:2000])

Professional with 11.3+ years of experience. I'm a marketing manager with substantial experience in cross-functional collaboration, stakeholder management, and execution. Lately I've been curious about how AI tools could augment my work — I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities. Project Manager | 11.3+ yrs experience Brand design and creative direction at a consumer-products company. Owned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social. Led the recent rebrand and managed a small external agency for production work. Comfortable across the Adobe suite, Figma, and the production side of brand and packaging design. Marketing leadership role at a B2B SaaS company. Owned the demand-generation function — content marketing, paid acquisition, SEO, email nurture

In [54]:
import re

technical_actions = [
    "built",
    "shipped",
    "deployed",
    "implemented",
    "designed",
    "developed",
]

technical_objects = [
    "model",
    "models",
    "ranking",
    "retrieval",
    "recommendation",
    "search",
    "pipeline",
    "pipelines",
    "embeddings",
    "vector database",
    "vector db",
    "ml system",
    "machine learning",
    "feature store",
    "xgboost",
    "lightgbm",
    "spark",
    "airflow",
    "faiss",
    "pinecone",
    "qdrant",
    "elasticsearch",
]

action_pattern = r"(?:" + "|".join(re.escape(action) for action in technical_actions) + r")"
object_pattern = r"(?:" + "|".join(re.escape(obj) for obj in technical_objects) + r")"
nearby_words = r"(?:\W+\w+){0,8}?\W+"

technical_production_pattern = re.compile(
    rf"(?<!\w)(?:{action_pattern}{nearby_words}{object_pattern}|{object_pattern}{nearby_words}{action_pattern})(?!\w)",
    flags=re.IGNORECASE,
)


def count_technical_production_phrases(text):
    return len(technical_production_pattern.findall(safe_text(text)))


technical_production_raw_score = df["candidate_text"].apply(count_technical_production_phrases)
max_technical_production_raw_score = technical_production_raw_score.max()
df["technical_production_score"] = (
    technical_production_raw_score / max_technical_production_raw_score
    if max_technical_production_raw_score
    else 0.0
)

df.sort_values(
    ["technical_production_score", "candidate_id"],
    ascending=[False, True],
).head(10)[["candidate_id", "technical_production_score"]]


,candidate_id,technical_production_score
5648,CAND_0005649,1.0
7411,CAND_0007412,1.0
30952,CAND_0030953,0.9
44882,CAND_0044883,0.9
10684,CAND_0010685,0.8
19479,CAND_0019480,0.8
30,CAND_0000031,0.7
7008,CAND_0007009,0.7
10769,CAND_0010770,0.7
17959,CAND_0017960,0.7


In [55]:
df["final_score_v2"] = (
    0.20 * df["experience_score"]
    + 0.20 * df["title_score"]
    + 0.10 * df["behavior_score"]
    + 0.10 * df["production_score"]
    + 0.15 * df["technical_production_score"]
    + 0.25 * df["retrieval_score"]
)

top_candidates_v2 = df.sort_values(
    "final_score_v2",
    ascending=False
)

top_candidates_v2[
[
    "candidate_id",
    "profile_current_title",
    "profile_years_of_experience",
    "final_score_v2"
]
].head(15)

,candidate_id,profile_current_title,profile_years_of_experience,final_score_v2
7008,CAND_0007009,Recommendation Systems Engineer,7.9,0.819917
41609,CAND_0041610,Recommendation Systems Engineer,6.7,0.798987
30,CAND_0000031,Recommendation Systems Engineer,6.0,0.749419
53694,CAND_0053695,Recommendation Systems Engineer,5.8,0.730857
41668,CAND_0041669,Recommendation Systems Engineer,8.0,0.729241
17959,CAND_0017960,Recommendation Systems Engineer,7.7,0.726487
18721,CAND_0018722,Recommendation Systems Engineer,6.6,0.719779
61264,CAND_0061265,Recommendation Systems Engineer,6.6,0.716962
7411,CAND_0007412,Applied ML Engineer,7.4,0.685941
14439,CAND_0014440,Recommendation Systems Engineer,6.4,0.677515


In [56]:
cid = "CAND_0000060"

row = df[df["candidate_id"] == cid]

row[
[
    "profile_current_title",
    "experience_score",
    "title_score",
    "behavior_score",
    "production_score",
    "technical_production_score",
    "retrieval_score",
    "final_score_v2"
]
]
# row.iloc[0]["candidate_text"][:1500]
row = df[df["candidate_id"] == "CAND_0000010"].iloc[0]

print("TITLE:", row["profile_current_title"])
print("EXP:", row["profile_years_of_experience"])
print()
print(row["candidate_text"][:2000])

TITLE: Data Engineer
EXP: 4.6

Software / data professional with 4.6 years of experience building data pipelines, backend systems, and analytics infrastructure. Most of my work has been data pipelines and analytics infrastructure; I've shipped a couple of small ML features but it's not the core of my day. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice. Data Engineer | Data pipelines & analytics Mixed data science and analytics-engineering role at a marketing-analytics startup. Spent maybe 30% of my time on lightweight ML (clustering, classification, churn prediction in sklearn/XGBoost) and 70% on data infrastructure and dashboards. Comfortable with the m

In [61]:
top20 = df.sort_values("final_score_v2", ascending=False)

top20[
    [
        "candidate_id",
        "profile_current_title",
        "profile_years_of_experience",
        "experience_score",
        "title_score",
        "behavior_score",
        "technical_production_score",
        "retrieval_score",
        "final_score_v2"
    ]
].head(20)

,candidate_id,profile_current_title,profile_years_of_experience,experience_score,title_score,behavior_score,technical_production_score,retrieval_score,final_score_v2
7008,CAND_0007009,Recommendation Systems Engineer,7.9,1.0,1.0,0.796,0.7,0.785714,0.819917
41609,CAND_0041610,Recommendation Systems Engineer,6.7,1.0,1.0,0.824,0.6,0.706349,0.798987
30,CAND_0000031,Recommendation Systems Engineer,6.0,1.0,1.0,0.718,0.7,0.468254,0.749419
53694,CAND_0053695,Recommendation Systems Engineer,5.8,1.0,1.0,0.730,0.6,0.515873,0.730857
41668,CAND_0041669,Recommendation Systems Engineer,8.0,1.0,1.0,0.844,0.5,0.523810,0.729241
17959,CAND_0017960,Recommendation Systems Engineer,7.7,1.0,1.0,0.740,0.7,0.404762,0.726487
18721,CAND_0018722,Recommendation Systems Engineer,6.6,1.0,1.0,0.738,0.4,0.603175,0.719779
61264,CAND_0061265,Recommendation Systems Engineer,6.6,1.0,1.0,0.622,0.5,0.563492,0.716962
7411,CAND_0007412,Applied ML Engineer,7.4,1.0,0.0,0.702,1.0,0.833333,0.685941
14439,CAND_0014440,Recommendation Systems Engineer,6.4,1.0,1.0,0.736,0.4,0.412698,0.677515


In [59]:
bottom20 = df.sort_values(
    "final_score_v2",
    ascending=True
)

bottom20[
    [
        "candidate_id",
        "profile_current_title",
        "profile_years_of_experience",
        "final_score_v2"
    ]
].head(20)

,candidate_id,profile_current_title,profile_years_of_experience,final_score_v2
68669,CAND_0068670,Graphic Designer,2.5,0.056800
87643,CAND_0087644,Graphic Designer,1.9,0.056800
79676,CAND_0079677,HR Manager,1.7,0.059800
90608,CAND_0090609,Content Writer,1.5,0.060600
3724,CAND_0003725,Content Writer,1.7,0.060800
57555,CAND_0057556,Accountant,1.2,0.061600
95982,CAND_0095983,Customer Support,1.6,0.061600
98334,CAND_0098335,Civil Engineer,1.6,0.062000
21963,CAND_0021964,HR Manager,2.1,0.062600
69340,CAND_0069341,HR Manager,2.3,0.062800


In [62]:
top100 = df.sort_values(
    "final_score_v2",
    ascending=False
).head(100)

top100["profile_current_title"].value_counts().head(20)

profile_current_title
Recommendation Systems Engineer     23
Backend Engineer                    21
Software Engineer                   18
Data Engineer                       14
Search Engineer                      4
Applied ML Engineer                  3
Staff Machine Learning Engineer      3
AI Engineer                          3
Senior Machine Learning Engineer     2
Senior Data Scientist                2
Senior NLP Engineer                  2
Lead AI Engineer                     2
NLP Engineer                         1
Machine Learning Engineer            1
Senior Applied Scientist             1
Name: count, dtype: int64